# Lesson 1
## Embeddings
A way to represent sentences as vectors in an embedding space(vector space).
Semantically similar vectors get placed near to eachother. 

Example:

These two sentences:
* “The dog is running”
* “A puppy is sprinting”

look different as text, but semantically they’re similar.
An embedding model maps both into nearby points in vector space. The cliser teh vectors teh similar the meaning they share.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The dog is running in the park",
    "A puppy is sprinting outside",
    "SQL is used for databases"
]

embeddings = model.encode(sentences)

query = model.encode(['A dog is running'])

scores = cosine_similarity(query, embeddings)
print(scores)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.66139054  0.69718087 -0.0452537 ]]


The 2nd embedded sentence shows the highest cosine_similarity score which means its most semantically similar to the query. 

In [2]:
query.shape #the model represented the sentence as a 384 dimension vector 

(1, 384)

In [3]:
# Test 2: this one shows how keyword search and embeddings search differ

sentences = [
    "The server crashed because memory was exhausted",
    "The RAM overflowed causing the machine to fail",
    "The word memory appears here but unrelated"
]

embeddings = model.encode(sentences)

query = model.encode(["system failed due to low memory"])

scores = cosine_similarity(query, embeddings)

for sentence, score in zip(sentences, scores[0]):
    print(sentence, '->', score)

The server crashed because memory was exhausted -> 0.6742977
The RAM overflowed causing the machine to fail -> 0.6393724
The word memory appears here but unrelated -> 0.39389783


Here the cosine_similarity ran linearly comparing the query embeddings to each of the sentence embeddings but,

* 10 chunks → easy
* 10,000 chunks → slower
* 1,000,000 chunks → needs indexing tricks

Thats why vector databases exist. It avoids brute-force scans by using approximate nearest-neighbor methods.

# Lesson 2
## Vector Databases and Chunking
**They store:**
* vectors
* metadata
* document IDs
* chunk text

**They handle:**
* fast nearest-neighbor lookup
* filtering
* persistence

### Chunking
Diving bigger text paragraphs into smaller size chunks.

In [4]:
def chunk_text(text, chunk_size=500):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += end

    return chunks

In [5]:
text = '''Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it takes, let’s dive into some
examples of the most important prompting techniques.
General prompting / zero shot
A zero-shot5
prompt is the simplest type of prompt. It only provides a description of a task
and some text for the LLM to get started with. This input could be anything: a question, a
start of a story, or instructions. The name zero-shot stands for ’no examples’.'''

In [6]:
import numpy as np

chunks = chunk_text(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] #top 2 best matches

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a res

lets introduce chunking with overlap by modifing out chunk_text method, for that we'll have to add a overlap attribute and a little tweak to the code should do it

In [7]:
def chunk_text_with_overlap(text, chunk_size=500, overlap=10):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [8]:
import numpy as np

chunks = chunk_text_with_overlap(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] 

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
ate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it 

Overlapping preserves continuity. Also a thing to notice is that teh score of the 2nd chunk increased which means overlapping helps preserve the continuity and flow over all the chunks.

# Lesson 3
## Real Document QA using LlamaIndex

In [9]:
# pip install llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 95.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
Note: you may need to restart the kernel to use updated packages.


In [10]:
# pip install llama-index-embeddings-huggingface

Note: you may need to restart the kernel to use updated packages.


In [11]:
# pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    

In [12]:
# !pip install pypdf

In [13]:
# !pip install llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00


In [14]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset = Path("/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs")

loader = PDFReader()

documents = []

for pdf in dataset.glob("*.pdf"):
    doc = loader.load_data(file=pdf)
    documents.extend(doc)
    
print(documents[0].text[:500])

Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



In [15]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex.from_documents(documents)

retriever = index.as_retriever(similarity_top_k=2)

nodes = retriever.retrieve("What is prompt engineering?")

for i, node in enumerate(nodes):
    print(f"\n--- Chunk {i+1} ---\n")
    print(node.text[:1000])

2026-05-16 17:56:01,461 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-05-16 17:56:01,580 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-16 17:56:01,595 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-05-16 17:56:01,613 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-05-16 17:56:01,687 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-16 17:56:01,702 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-16 17:56:01,718 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

2026-05-16 17:56:01,787 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-16 17:56:01,802 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-16 17:56:01,863 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-16 17:56:01,878 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
2026-05-16 17:56:01,899 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

2026-05-16 17:56:06,317 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-16 17:56:06,451 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-16 17:56:06,467 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-16 17:56:06,484 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-05-16 17:56:06,569 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-05-16 17:56:06,638 - INFO - 1 prompt is loaded, with the key: query



--- Chunk 1 ---

Foundational Large Language Models & Text Generation
52
February 2025
Using large language models
Prompt engineering and sampling techniques have a strong influence on the performance of 
LLMs. Prompt engineering is the process of designing and refining the text inputs (prompts) 
that you feed into an LLM to achieve desired and relevant outputs. Sampling techniques 
determine the way in which output tokens are chosen and influence the correctness, 
creativity and diversity of the resulting output. We next discuss different variants of prompt 
engineering and sampling techniques as well as touch on some important parameters that 
can have a significant impact on LLM performance.
Prompt engineering 
LLMs are very powerful, but they still need guidance to unleash their full potential. Prompt 
engineering is a critical component in guiding an LLM to yield desired outputs. This might 
include grounding the model to yield factual responses or unleashing the creativity of th

# Lesson 4
## Permanent storage (ChromaDB)

In [16]:
# !pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp protobuf
# !pip install chromadb==0.5.5 \
#     opentelemetry-api==1.27.0 \
#     opentelemetry-sdk==1.27.0 \
#     opentelemetry-exporter-otlp==1.27.0 \
#     protobuf==4.25.3 -q

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Found existing installation: opentelemetry-api 1.41.1
Uninstalling opentelemetry-api-1.41.1:
  Successfully uninstalled opentelemetry-api-1.41.1
Found existing installation: opentelemetry-sdk 1.41.1
Uninstalling opentelemetry-sdk-1.41.1:
  Successfully uninstalled opentelemetry-sdk-1.41.1
Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 11.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.6 MB/s eta 0:0

In [17]:
# !pip install -U chromadb==0.5.23 llama-index-vector-stores-chroma==0.5.5 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 10.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [18]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

db = chromadb.PersistentClient(path='./chroma_db')
collection = db.get_or_create_collection('whitepaper')

2026-05-16 17:56:38,655 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-05-16 17:56:39,141 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-16 17:56:39,170 - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


**Meaning**
* creates folder chroma_db
* collection/db name = whitepapers
* saved permanently

In [19]:
#connect to llama_index

vector_store = ChromaVectorStore(chroma_collection=collection)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

now llama index writes into chroma

In [20]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

2026-05-16 17:56:42,204 - ERROR - Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [21]:
db = chromadb.PersistentClient(path="./chroma_db")

collection = db.get_collection("whitepaper")
print(collection.count())

2026-05-16 17:56:42,217 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


260


## Metadata Filtering

In [22]:
files = set()

for doc in documents:
    files.add(doc.metadata['file_name'])

print(files)
print(len(files))

{'whitepaper 1.pdf', 'whitepaper_emebddings_vectorstores_v2.pdf', '22365_19_Agents_v8.pdf', '22365_3_Prompt Engineering_v7 (1).pdf'}
4


In [28]:
retriever = index.as_retriever(
    similarity_top_k=2,
    fiters={
    'file_name':'22365_19_Agents_v8.pdf'
    }
)

nodes = retriever.retrieve('What are agents?')

for node in nodes:
    print(node.metadata)
    print(node.text[:800])
    print("="*50)

2026-05-16 18:04:38,342 - ERROR - Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'page_label': '5', 'file_name': '22365_19_Agents_v8.pdf'}
Agents
5
February 2025
What is an agent?
In its most fundamental form, a Generative AI agent can be defined as an application that 
attempts to achieve a goal by observing the world and acting upon it using the tools that it 
has at its disposal. Agents are autonomous and can act independently of human intervention, 
especially when provided with proper goals or objectives they are meant to achieve. Agents 
can also be proactive in their approach to reaching their goals. Even in the absence of 
explicit instruction sets from a human, an agent can reason about what it should do next to 
achieve its ultimate goal. While the notion of agents in AI is quite general and powerful, this 
whitepaper focuses on the specific types of agents that Generative AI models are capable of 
building at the t
{'page_label': '40', 'file_name': '22365_19_Agents_v8.pdf'}
Agents
40
February 2025
Summary
In this whitepaper we’ve discussed the foundatio

In [29]:
for node in nodes:
    print(node.score)

0.668149476739516
0.563455225204821


Whenever chunks are retrieved check for:
* Relevance - Does teh chunk acctually contain some answerable info
* Completeness - Is the answer split across multiple chunks
* Noise - Does the retrieved chunk contain unrelated surrounding text

This is to be done because we dont want a system that answered we want a system that gave the correct answer. Thats why
* tuning chunking
* testing retrieval quality
* finding failure cases

is the real engineering part.